###  Create a Messy Dataset (for testing)

In [1]:
import pandas as pd
import numpy as np
import os

# Build a messy dataset
data = [
    ["  John Smith ", "  Engineering ", "2023-15-01", "M", "85000", "john@x.com"],
    ["JANE DOE", "engineering", "01/22/2022", "F", "90000", "jane@x.com"],
    ["bob  brown", " Marketing", "March 3, 2021", "m", "72,500", "bob@x.com"],
    ["Alice Green", "MARKETING  ", "2020/11/30", "F", "not_available", "alice@x.com"],
    ["   ", "HR", "2024-02-31", "f", "65000", "  "],
    ["Carlos Ruiz", "hr ", "15-06-2023", "M", "78000", "carlos@x.com"],
    ["  Eva  ", "Finance", "2022.07.19", "F", "95,000", "eva@x.com"],
    ["Frank Liu", "FINANCE", "2021-13-45", "m", "88000", None],
]

df_raw = pd.DataFrame(data, columns=[
    "name", "department", "hire_date", "gender", "salary", "email"
])

# Save with a weird encoding to simulate real-world issue
df_raw.to_csv("employees_messy.csv", index=False, encoding="utf-8-sig")
print("Messy file written.")

Messy file written.


### Load (handle encoding + delimiter + dtype)

In [2]:
# Try loading with UTF-8 first, fall back to latin-1
def smart_load(path, **kwargs):
    for enc in ("utf-8-sig", "utf-8", "latin-1"):
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except UnicodeDecodeError:
            print(f"⚠️  {enc} failed, trying next...")
    raise RuntimeError("Could not decode file with any encoding.")

# Peek first few rows BEFORE loading everything (nrows trick)
print("---- Peek (first 3 rows) ----")
print(pd.read_csv("employees_messy.csv", nrows=3, encoding="utf-8-sig"))

print("\n---- Full load ----")
df = smart_load(
    "employees_messy.csv",
    sep=",",
    dtype={          # explicit dtype = no silent casting
        "name": "string",
        "department": "string",
        "hire_date": "string",   # keep dates as string, convert later
        "gender": "string",
        "salary": "string",      # salary is dirty → keep as string
        "email": "string",
    },
    keep_default_na=True,
)
print(df)

---- Peek (first 3 rows) ----
            name      department      hire_date gender  salary       email
0    John Smith     Engineering      2023-15-01      M   85000  john@x.com
1       JANE DOE     engineering     01/22/2022      F   90000  jane@x.com
2     bob  brown       Marketing  March 3, 2021      m  72,500   bob@x.com

---- Full load ----
            name      department      hire_date gender         salary  \
0    John Smith     Engineering      2023-15-01      M          85000   
1       JANE DOE     engineering     01/22/2022      F          90000   
2     bob  brown       Marketing  March 3, 2021      m         72,500   
3    Alice Green     MARKETING       2020/11/30      F  not_available   
4                             HR     2024-02-31      f          65000   
5    Carlos Ruiz             hr      15-06-2023      M          78000   
6          Eva           Finance     2022.07.19      F         95,000   
7      Frank Liu         FINANCE     2021-13-45      m          8

### Inspect

In [3]:
print("\n===== Shape =====")
print(df.shape)

print("\n===== dtypes =====")
print(df.dtypes)

print("\n===== Info =====")
df.info()

print("\n===== Missing values =====")
print(df.isna().sum())

print("\n===== Sample values per categorical column =====")
for col in ["name", "department", "gender"]:
    print(f"\n{col}:")
    print(df[col].dropna().unique()[:10])

print("\n===== date-like columns =====")
print(df["hire_date"].tolist())


===== Shape =====
(8, 6)

===== dtypes =====
name          string[python]
department    string[python]
hire_date     string[python]
gender        string[python]
salary        string[python]
email         string[python]
dtype: object

===== Info =====
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   name        8 non-null      string
 1   department  8 non-null      string
 2   hire_date   8 non-null      string
 3   gender      8 non-null      string
 4   salary      8 non-null      string
 5   email       7 non-null      string
dtypes: string(6)
memory usage: 516.0 bytes

===== Missing values =====
name          0
department    0
hire_date     0
gender        0
salary        0
email         1
dtype: int64

===== Sample values per categorical column =====

name:
<StringArray>
['  John Smith ',      'JANE DOE',    'bob  brown',   'Alice Green',
       

### Clean

In [4]:
df_clean = df.copy()

# 1) Normalise text (strip whitespace, collapse inner spaces, drop empties)
def clean_text(s: pd.Series) -> pd.Series:
    return (
        s.astype("string")
         .str.strip()
         .str.replace(r"\s+", " ", regex=True)
         .replace("", pd.NA)
    )

for col in ["name", "department", "gender", "email"]:
    df_clean[col] = clean_text(df_clean[col])

# 2) Standardise categorical values (Title case)
df_clean["department"] = df_clean["department"].str.title()
df_clean["gender"]     = df_clean["gender"].str.upper().map(
    {"M": "Male", "F": "Female"}
)

# 3) Convert dates safely
df_clean["hire_date"] = pd.to_datetime(
    df_clean["hire_date"],
    errors="coerce",
    dayfirst=False,       # data is mostly US-style
    format="mixed",       # pandas ≥ 2.0 handles mixed formats
)

# 4) Salary → numeric (strip commas and anything non-digit)
df_clean["salary"] = pd.to_numeric(
    df_clean["salary"].astype("string")
        .str.replace(r"[,\s$]", "", regex=True),
    errors="coerce",
)

# 5) Drop rows where every key field is missing
key_fields = ["name", "department", "hire_date", "salary"]
before = len(df_clean)
df_clean = df_clean.dropna(subset=key_fields, how="all")
print(f"Dropped {before - len(df_clean)} empty row(s).")

# 6) Flag rows with any remaining problems (for the audit)
df_clean["has_issues"] = df_clean.isna().any(axis=1)

print(df_clean)

Dropped 0 empty row(s).
          name   department  hire_date  gender  salary         email  \
0   John Smith  Engineering        NaT    Male   85000    john@x.com   
1     JANE DOE  Engineering 2022-01-22  Female   90000    jane@x.com   
2    bob brown    Marketing 2021-03-03    Male   72500     bob@x.com   
3  Alice Green    Marketing 2020-11-30  Female    <NA>   alice@x.com   
4         <NA>           Hr        NaT  Female   65000          <NA>   
5  Carlos Ruiz           Hr 2023-06-15    Male   78000  carlos@x.com   
6          Eva      Finance 2022-07-19  Female   95000     eva@x.com   
7    Frank Liu      Finance        NaT    Male   88000          <NA>   

   has_issues  
0        True  
1       False  
2       False  
3        True  
4        True  
5       False  
6       False  
7        True  


### Review (audit before export)

In [5]:
print("\n===== Final dtypes =====")
print(df_clean.dtypes)

print("\n===== Missing values after cleaning =====")
print(df_clean.isna().sum())

print("\n===== Unique categoricals after cleaning =====")
for col in ["department", "gender"]:
    print(f"{col}: {sorted(df_clean[col].dropna().unique().tolist())}")

print("\n===== Rows with remaining issues =====")
print(df_clean[df_clean["has_issues"]])

# Sanity: no negative salaries, dates within a plausible range
assert (df_clean["salary"].dropna() > 0).all(), "Negative salary found!"
assert df_clean["hire_date"].dropna().between("1990-01-01", "2030-12-31").all()
print("\n✅ Audit checks passed.")


===== Final dtypes =====
name          string[python]
department    string[python]
hire_date     datetime64[ns]
gender                object
salary                 Int64
email         string[python]
has_issues              bool
dtype: object

===== Missing values after cleaning =====
name          1
department    0
hire_date     3
gender        0
salary        1
email         2
has_issues    0
dtype: int64

===== Unique categoricals after cleaning =====
department: ['Engineering', 'Finance', 'Hr', 'Marketing']
gender: ['Female', 'Male']

===== Rows with remaining issues =====
          name   department  hire_date  gender  salary        email  \
0   John Smith  Engineering        NaT    Male   85000   john@x.com   
3  Alice Green    Marketing 2020-11-30  Female    <NA>  alice@x.com   
4         <NA>           Hr        NaT  Female   65000         <NA>   
7    Frank Liu      Finance        NaT    Male   88000         <NA>   

   has_issues  
0        True  
3        True  
4        Tru

### Export

In [6]:
df_clean.drop(columns="has_issues").to_csv(
    "employees_clean.csv", index=False, encoding="utf-8"
)
print("✅ Clean file exported to employees_clean.csv")

# Optional: also export a small audit report
audit = {
    "rows_before": len(df),
    "rows_after":  len(df_clean),
    "cols":        len(df_clean.columns),
    "missing_after": int(df_clean.isna().sum().sum()),
}
pd.Series(audit).to_csv("audit_report.csv", header=False)
print(pd.Series(audit))

✅ Clean file exported to employees_clean.csv
rows_before      8
rows_after       8
cols             7
missing_after    7
dtype: int64


To download from Colab

In [7]:
from google.colab import files
files.download("employees_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>